# 07 · As queries dos slides

Cada célula roda uma query mostrada nos slides, na mesma ordem da apresentação. Use durante a aula para mostrar o resultado ao vivo, ou depois para revisar.

Precisa só dos dados do notebook `00`. Tudo roda num banco em memória, então nada do `curso.duckdb` é alterado.

Onde o slide encurta nomes de colunas para caber na tela (`qtd`, `preco`, `data`), a célula usa os nomes reais (`quantidade`, `preco_unitario`, `data_venda`) e avisa no comentário.

In [ ]:
import duckdb, os
import pandas as pd

con = duckdb.connect()  # banco em memória

# tabelas usadas pelos slides
con.sql("""
    CREATE VIEW vendas AS FROM 'dados/vendas.parquet';
    CREATE TABLE produtos AS SELECT produto_id, nome, categoria, preco FROM 'dados/produtos.json';
    CREATE VIEW precos_historico AS FROM 'dados/precos_historico.parquet';
""")
con.sql("SHOW TABLES").show()

## Slide · Primeira consulta
O caminho do arquivo entra direto no `FROM`.

In [ ]:
duckdb.sql("""
    SELECT canal, count(*) AS itens,
           round(sum(quantidade * preco_unitario) / 1e6, 2) AS receita_mi
    FROM 'dados/vendas.parquet'
    GROUP BY canal ORDER BY receita_mi DESC
""").show()

In [ ]:
# um DataFrame do pandas entra no FROM pelo nome da variável
metas = pd.DataFrame({"canal": ["site", "app", "loja_fisica", "marketplace"],
                      "meta_mi": [650, 720, 690, 700]})

duckdb.sql("""
    SELECT v.canal,
           round(sum(quantidade * preco_unitario) / 1e6, 2) AS receita_mi,
           any_value(m.meta_mi) AS meta_mi
    FROM 'dados/vendas.parquet' v
    JOIN metas m USING (canal)
    GROUP BY v.canal ORDER BY receita_mi DESC
""").show()

In [ ]:
# o resultado volta no formato que você usa
r = duckdb.sql("SELECT canal, count(*) AS itens FROM 'dados/vendas.parquet' GROUP BY canal")
print(type(r.df()))
print(type(r.pl()))
print(type(r.arrow()))

## Slide · Arquivos de qualquer lugar

In [ ]:
# vários arquivos com glob, e de onde veio cada linha
con.sql("""
    SELECT filename, count(*)
    FROM read_json('dados/logs/*.jsonl', filename = true)
    GROUP BY ALL
""").show()

In [ ]:
# partições Hive: pula pastas inteiras
con.sql("""
    FROM read_parquet('dados/lake/vendas/*/*/*.parquet', hive_partitioning = true)
    WHERE ano = 2025 AND mes <= 3
""").show(max_rows=5)

In [ ]:
# CSV para Parquet em uma linha
con.sql("COPY (FROM 'dados/vendas_2025_01.csv') TO 'dados/vendas_2025_01_slide.parquet' (FORMAT parquet)")
con.sql("SELECT count(*) AS linhas FROM 'dados/vendas_2025_01_slide.parquet'").show()

## Slide · SQL amigável

In [ ]:
con.sql("FROM vendas LIMIT 5").show()

In [ ]:
# no slide: round(preco) AS preco
con.sql("""
    SELECT * EXCLUDE (venda_id)
             REPLACE (round(preco_unitario) AS preco_unitario)
    FROM vendas
    LIMIT 5
""").show()

In [ ]:
# no slide: COLUMNS('qtd|preco')
con.sql("SELECT max(COLUMNS('quantidade|preco')) FROM vendas").show()

In [ ]:
con.sql("SUMMARIZE produtos").show()

In [ ]:
# alias como prefixo (1.2), reaproveitado na mesma query
# no slide: data, qtd, preco
con.sql("""
    SELECT
      mes:     date_trunc('month', data_venda),
      receita: sum(quantidade * preco_unitario),
      ticket:  receita / count(DISTINCT pedido_id)
    FROM vendas
    GROUP BY ALL
    ORDER BY ALL
""").show(max_rows=6)

## Slide · `QUALIFY` e `PIVOT`

In [ ]:
# produto mais vendido de cada categoria (no slide: qtd)
con.sql("""
    SELECT categoria, nome,
           sum(quantidade) AS unidades
    FROM vendas JOIN produtos
      USING (produto_id)
    GROUP BY categoria, nome
    QUALIFY row_number() OVER (
      PARTITION BY categoria
      ORDER BY unidades DESC) = 1
""").show()

In [ ]:
# vendas_2025 com as colunas do slide: categoria, trimestre, valor
con.sql("""
    CREATE VIEW vendas_2025 AS
    SELECT categoria,
           'T' || quarter(data_venda) AS trimestre,
           quantidade * preco_unitario AS valor
    FROM vendas JOIN produtos USING (produto_id)
    WHERE year(data_venda) = 2025
""")

con.sql("""
    PIVOT vendas_2025
    ON trimestre
    USING sum(valor)
    GROUP BY categoria
""").show()

## Slide · `ASOF JOIN`
Para cada venda, o preço de tabela que valia na data. A célula limita a 10 linhas para não imprimir 5 milhões.

In [ ]:
con.sql("""
    SELECT v.data_venda, v.preco_unitario,
           h.valido_desde, h.preco_tabela
    FROM vendas v
    ASOF JOIN precos_historico h
      ON v.produto_id = h.produto_id
     AND v.data_venda >= h.valido_desde
    LIMIT 10
""").show()

## Slide · `MERGE INTO`
Primeiro, um catálogo com 5 produtos e uma atualização: dois preços novos e um produto que ainda não existe.

In [ ]:
con.sql("""
    CREATE TABLE catalogo AS
    SELECT produto_id, nome, preco FROM produtos WHERE produto_id <= 5;

    CREATE TABLE atualizacao AS
    SELECT * FROM (VALUES
        (1, 'Produto 1', 10.0),
        (2, 'Produto 2', 20.0),
        (999, 'Produto novo', 99.9)
    ) t(produto_id, nome, preco);
""")

con.sql("""
    MERGE INTO catalogo c
    USING atualizacao a USING (produto_id)
    WHEN MATCHED THEN UPDATE
      SET preco = a.preco
    WHEN NOT MATCHED THEN INSERT
      VALUES (a.produto_id, a.nome, a.preco)
    RETURNING merge_action, *
""").show()

## Slide · Extensões
Precisa de internet e de credenciais da AWS. Sem isso, a célula mostra o erro e segue.

In [ ]:
try:
    con.sql("INSTALL httpfs; LOAD httpfs;")
    con.sql("CREATE SECRET (TYPE s3, PROVIDER credential_chain)")
    con.sql("FROM 's3://meu-bucket/vendas/*.parquet'").show()  # troque pelo seu bucket
except Exception as e:
    print("Não rodou aqui:", type(e).__name__, str(e)[:200])

## Slide · Triggers (2.0)

In [ ]:
con.sql("""
    CREATE TABLE auditoria_preco (
        produto_id INTEGER, preco_antigo DOUBLE, preco_novo DOUBLE, alterado_em TIMESTAMP
    );

    CREATE TRIGGER trg_auditoria_preco
    AFTER UPDATE ON produtos
    REFERENCING OLD TABLE AS antigo NEW TABLE AS novo
    FOR EACH STATEMENT
    INSERT INTO auditoria_preco
      SELECT antigo.produto_id, antigo.preco, novo.preco, now()
      FROM antigo JOIN novo USING (produto_id)
      WHERE antigo.preco <> novo.preco;
""")

# dispara o trigger
con.sql("UPDATE produtos SET preco = round(preco * 1.10, 2) WHERE produto_id IN (1, 2, 3)")
con.sql("FROM auditoria_preco").show()

In [ ]:
_ = con.sql("DROP TRIGGER trg_auditoria_preco ON produtos")

## Slide · `VARIANT` e CTEs que modificam dados (2.0)

In [ ]:
con.sql("""
    CREATE TABLE eventos AS
    SELECT * FROM (VALUES
        (1, {'tipo': 'clique', 'botao': 'comprar'}::VARIANT),
        (2, {'tipo': 'compra', 'valor': 199.9, 'itens': [10, 20]}::VARIANT),
        (3, 'texto livre'::VARIANT),
        (4, 42::VARIANT)
    ) t(id, payload)
""")

con.sql("""
    SELECT id,
      variant_typeof(payload),
      variant_keys(payload),
      payload.tipo,
      payload.valor::DOUBLE
    FROM eventos
""").show()

In [ ]:
# move as vendas canceladas de janeiro para outra tabela, numa instrução só
con.sql("""
    CREATE TABLE vendas_jan AS FROM 'dados/vendas_2025_01.csv';
    CREATE TABLE canceladas AS FROM vendas_jan LIMIT 0;
""")

con.sql("""
    WITH removidas AS (
      DELETE FROM vendas_jan
      WHERE status = 'cancelado'
      RETURNING *
    ), arquivadas AS (
      INSERT INTO canceladas
      SELECT * FROM removidas
      RETURNING venda_id
    ) SELECT count(*) FROM arquivadas
""").show()

## Slide · Quack e UI
Esses dois slides rodam no terminal, não no notebook.

**Quack** (servidor e cliente DuckDB). Num terminal:
```
$ duckdb curso.duckdb
CALL quack_serve(token = 'aula');
```
Em outro terminal:
```
$ duckdb
ATTACH 'quack:localhost' AS remoto (TOKEN 'aula');
CONNECT remoto;
SELECT count(*) FROM vendas;
```

**UI**:
```
$ duckdb -ui curso.duckdb
```
Ou, de dentro do Python, `con.sql("CALL start_ui()")`, que abre http://localhost:4213. O notebook `04` tem esse passo.

## Slide · Caso: logs de aplicação
Rotas e servidores com taxa de erro 5xx acima do dobro da média.

In [ ]:
con.sql("""
    SELECT rota, servidor, count(*) AS reqs,
      100 * avg((status >= 500)::INT) AS pct_5xx
    FROM 'dados/logs/*.jsonl'
    GROUP BY rota, servidor
    QUALIFY pct_5xx > 2 * avg(pct_5xx) OVER ()
    ORDER BY pct_5xx DESC
""").show()